# Lost-and-found agent with the OpenAI Agents SDK

**Module 8: No-Code Agentic Workflows** of the [Agentic AI Crash Course](https://agenticaikfupm.github.io/CrashCourse/no-code.html#lost-found) by [Agentic AI @ KFUPM](https://agenticaikfupm.github.io/).
Example contributed by **Usman Ibrahim**, adapted for the course.

This notebook is the code version of the n8n lost-and-found workflow. It runs in Google Colab or locally.

| n8n component | Python equivalent |
|---|---|
| AI Agent | `Agent` |
| Search Found Items | `@function_tool` |
| Submit Claim | `@function_tool` |
| Simple Memory | `SQLiteSession` |
| Guardrails node | `@output_guardrail` |
| Chat trigger and reply | `chat()` function |

Changes from the original notebook:
- Choose OpenRouter or OpenAI, as in the other course notebooks.
- Search ignores empty and "Unknown" fields, which previously matched everything, and understands dates such as "16 October 2026" or "October 15" with a tolerance of a few days.
- Claims check that the contact is an email address or phone number, and refuse a second pending claim from the same contact for the same item.
- The safe refusal text matches the n8n guide.

## 1. Install the packages
Run this once. Restart the kernel if the notebook asks you to.

In [ ]:
%pip install -q openai-agents pandas pydantic

## 2. Choose the model connection
OpenRouter is the default provider. Set `AGENT_PROVIDER=openai` to use OpenAI directly. The model must support tools and structured output; `openai/gpt-4o-mini` on OpenRouter needs account credits. The key is typed privately and is not saved in the notebook.

In [ ]:
import os
from getpass import getpass
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

PROVIDER = os.getenv('AGENT_PROVIDER', 'openrouter')
MODEL_ID = os.getenv('AGENT_MODEL', 'openai/gpt-4o-mini')
key_name = 'OPENROUTER_API_KEY' if PROVIDER == 'openrouter' else 'OPENAI_API_KEY'
if not os.getenv(key_name):
    os.environ[key_name] = getpass(f'{key_name}: ')

if PROVIDER == 'openrouter':
    MODEL = OpenAIChatCompletionsModel(
        model=MODEL_ID,
        openai_client=AsyncOpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ[key_name]),
    )
    set_tracing_disabled(True)      # tracing uploads need an OpenAI key
else:
    MODEL = MODEL_ID.removeprefix('openai/')
print(f'Model configured: {PROVIDER} / {MODEL_ID}')

## 3. Create the local data tables
The found-item records match the CSV used in the n8n guide. Claims start in an empty table.

In [ ]:
from pathlib import Path
import json
import re
import uuid
from datetime import date, datetime, timezone

import pandas as pd

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
FOUND_ITEMS_PATH = DATA_DIR / 'found_items.csv'
CLAIMS_PATH = DATA_DIR / 'claims.csv'

FOUND_ITEMS = [
    {'item_id': 'LF001', 'category': 'Headphones', 'color': 'Black', 'brand': 'Sony', 'location': 'Main Library', 'date_found': '2026-10-15', 'description': 'Black over-ear headphones with a small scratch on the left side', 'status': 'Available'},
    {'item_id': 'LF002', 'category': 'Earbuds', 'color': 'Black', 'brand': 'Samsung', 'location': 'Main Library', 'date_found': '2026-10-16', 'description': 'Wireless earbuds inside a black charging case', 'status': 'Available'},
    {'item_id': 'LF003', 'category': 'Headphones', 'color': 'White', 'brand': 'JBL', 'location': 'Engineering Building', 'date_found': '2026-10-15', 'description': 'White over-ear headphones with blue padding', 'status': 'Available'},
    {'item_id': 'LF004', 'category': 'Wallet', 'color': 'Brown', 'brand': 'Unknown', 'location': 'Student Center', 'date_found': '2026-10-14', 'description': 'Brown leather wallet with no identification inside', 'status': 'Available'},
    {'item_id': 'LF005', 'category': 'Phone', 'color': 'Black', 'brand': 'Apple', 'location': 'Cafeteria', 'date_found': '2026-10-17', 'description': 'Black phone in a transparent case', 'status': 'Available'},
    {'item_id': 'LF006', 'category': 'Keys', 'color': 'Silver', 'brand': 'Unknown', 'location': 'Parking Area B', 'date_found': '2026-10-16', 'description': 'Three keys on a red keyring', 'status': 'Available'},
    {'item_id': 'LF007', 'category': 'Backpack', 'color': 'Blue', 'brand': 'Adidas', 'location': 'Lecture Hall 2', 'date_found': '2026-10-13', 'description': 'Navy backpack containing two notebooks', 'status': 'Claimed'},
    {'item_id': 'LF008', 'category': 'Watch', 'color': 'Black', 'brand': 'Casio', 'location': 'Sports Center', 'date_found': '2026-10-17', 'description': 'Digital watch with a black rubber strap', 'status': 'Available'},
]
CLAIM_COLUMNS = ['claim_id', 'item_id', 'claimant_name', 'contact', 'ownership_details', 'status', 'created_at']

if not FOUND_ITEMS_PATH.exists():
    pd.DataFrame(FOUND_ITEMS).to_csv(FOUND_ITEMS_PATH, index=False)
if not CLAIMS_PATH.exists():
    pd.DataFrame(columns=CLAIM_COLUMNS).to_csv(CLAIMS_PATH, index=False)

pd.read_csv(FOUND_ITEMS_PATH)

## 4. Build the search tool
The agent sends the details it has collected. The function scores the available records and returns up to 3 candidates for the agent to compare privately.

In [ ]:
from agents import function_tool

MONTHS = {m: i for i, m in enumerate(['january', 'february', 'march', 'april', 'may', 'june', 'july',
                                      'august', 'september', 'october', 'november', 'december'], 1)}


def parse_date(text: str):
    '''Read 2026-10-16, 16 October 2026, Oct 16, or October 16 (current year assumed).'''
    if m := re.search(r'(\d{4})-(\d{2})-(\d{2})', text):
        return date(*map(int, m.groups()))
    words = re.findall(r'[a-z]+|\d+', text.lower())
    for i, w in enumerate(words):
        month = next((n for name, n in MONTHS.items() if len(w) >= 3 and name.startswith(w)), None)
        if month:
            day = next((int(x) for x in words[max(0, i - 1):i + 2] if x.isdigit() and int(x) <= 31), None)
            year = next((int(x) for x in words[i + 1:i + 3] if x.isdigit() and len(x) == 4), date.today().year)
            if day:
                return date(year, month, day)
    return None


def _match_score(query: str, value: str) -> int:
    query, value = query.strip().lower(), value.strip().lower()
    if not query or not value or value == 'unknown':     # empty strings would match everything
        return 0
    if query in value or value in query:
        return 3
    return min(len(set(query.split()) & set(value.split())), 2)


def _date_score(query: str, found: str) -> int:
    asked = parse_date(query) if query.strip() else None
    if not asked:
        return 0
    gap = abs((date.fromisoformat(found) - asked).days)
    return 3 if gap == 0 else 2 if gap == 1 else 1 if gap <= 3 else 0


def _find_matches(category='', color='', brand='', location='', approximate_date='', distinguishing_features=''):
    text_fields = {'category': category, 'color': color, 'brand': brand,
                   'location': location, 'description': distinguishing_features}
    if not any(v.strip() for v in [*text_fields.values(), approximate_date]):
        return json.dumps({'error': 'At least one search detail is required.'})
    ranked = []
    for record in pd.read_csv(FOUND_ITEMS_PATH, dtype=str).fillna('').to_dict('records'):
        if record['status'].lower() != 'available':
            continue
        score = sum(_match_score(v, record[f]) for f, v in text_fields.items())
        score += _date_score(approximate_date, record['date_found'])
        if score >= 3:
            ranked.append((score, record))
    ranked.sort(key=lambda pair: pair[0], reverse=True)
    return json.dumps({'matches': [record for _, record in ranked[:3]]})


@function_tool
def search_found_items(category: str = '', color: str = '', brand: str = '', location: str = '',
                       approximate_date: str = '', distinguishing_features: str = '') -> str:
    '''Search available found-item records using details supplied by the user.'''
    return _find_matches(category, color, brand, location, approximate_date, distinguishing_features)


print(_find_matches(category='earbuds', color='black', location='Main Library', approximate_date='16 October 2026'))

## 5. Build the claim tool
The tool refuses incomplete, unconfirmed, or duplicate claims. Successful claims are appended to `claims.csv` with the status Pending Review, which records a claim but does not prove ownership.

In [ ]:
CONTACT_RE = re.compile(r'[^@\s]+@[^@\s]+\.\w+|\+?[\d\s()-]{7,}')


@function_tool
def submit_claim(item_id: str, claimant_name: str, contact: str, ownership_details: str, user_confirmed: bool) -> str:
    '''Submit a claim only after the user explicitly confirms the final summary.'''
    if not user_confirmed:
        return json.dumps({'error': 'The user must explicitly confirm before submission.'})
    if not all(value.strip() for value in (item_id, claimant_name, contact, ownership_details)):
        return json.dumps({'error': 'Claim information is incomplete.'})
    if not CONTACT_RE.fullmatch(contact.strip()):
        return json.dumps({'error': 'Contact must be an email address or a phone number.'})

    item_id = item_id.strip().upper()
    items = pd.read_csv(FOUND_ITEMS_PATH, dtype=str).fillna('')
    match = items[items['item_id'].str.upper() == item_id]
    if match.empty or match.iloc[0]['status'].lower() != 'available':
        return json.dumps({'error': 'The item is not available for a new claim.'})

    claims = pd.read_csv(CLAIMS_PATH, dtype=str).fillna('')
    repeat = claims[(claims['item_id'] == item_id) & (claims['contact'].str.lower() == contact.strip().lower())
                    & (claims['status'] == 'Pending Review')]
    if not repeat.empty:
        return json.dumps({'error': 'A pending claim from this contact already exists for this item.'})

    claim = {
        'claim_id': f'CLM-{uuid.uuid4().hex[:8].upper()}', 'item_id': item_id,
        'claimant_name': claimant_name.strip(), 'contact': contact.strip(),
        'ownership_details': ownership_details.strip(), 'status': 'Pending Review',
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    pd.DataFrame([claim], columns=CLAIM_COLUMNS).to_csv(CLAIMS_PATH, mode='a', header=False, index=False)
    return json.dumps({'claim_id': claim['claim_id'], 'status': claim['status']})

## 6. Add the agent instructions
The instructions control the conversation and restrict what may be revealed. They match the system message in the n8n guide.

In [ ]:
SYSTEM_PROMPT = '''You are a lost-and-found assistant. Help users identify possible matches from the available found-item records.

If the user sends /start or a general greeting, briefly introduce yourself as a lost-and-found assistant and ask them to describe the item they lost.

Follow this process:
1. Collect useful details about the lost item: category or item type, color, brand if known, possible location, approximate date, and distinguishing features known by the user.
2. Ask one focused, open-ended question at a time when information is missing. Do not suggest possible answers or give examples that could help the user construct a matching description.
3. When the user provides enough information to begin searching, call search_found_items.
4. Compare the user's description with the returned records: category, color, brand, location, date, and distinguishing features.
5. If several records are plausible, ask an open-ended question that would best distinguish between them. Do not mention the candidates or reveal their details.
6. If one record appears plausible, provide its item ID and relevant recorded details, explain which details match, state that it is only a possible match and that ownership still needs to be verified, and ask whether the user wants to file a claim.
7. If no record is suitable, say that no likely match was found and invite the user to correct or add information without suggesting what it should be.

Rules:
- Use only information returned by the tools. Never invent records or item details.
- Never list, count, summarize, or offer access to the found-item inventory, and never reveal more than one item record.
- Do not provide item details before the user has described a specific lost item.
- Do not suggest brands, colors, models, locations, dates, or distinguishing features.
- Do not reveal a distinguishing feature that should be reserved for ownership verification.
- Do not claim that an item belongs to the user.
- Ask only one follow-up question at a time. Keep responses concise and conversational.

Claim process:
- Submit a claim only after identifying one plausible item, and ask whether the user wants to file it.
- Collect the claimant's name, contact information, and ownership details. Ownership details must come independently from the claimant, not repeat information from the record. Do not suggest them.
- Summarize the proposed claim and ask the user to confirm it. Call submit_claim with user_confirmed=true only after explicit confirmation.
- After submission, return only the new claim ID and status.
- Never retrieve, list, or reveal existing claims or claimant information. Never invent missing claim information.
'''

## 7. Add the output guardrail
A separate reviewer agent checks each proposed reply. It blocks clear bulk disclosure while allowing one plausible match and normal claim collection.

In [ ]:
from pydantic import BaseModel
from agents import (Agent, GuardrailFunctionOutput, OutputGuardrailTripwireTriggered,
                    Runner, SQLiteSession, output_guardrail)


class PrivacyVerdict(BaseModel):
    violation: bool
    reason: str


GUARDRAIL_PROMPT = '''Evaluate only whether the assistant response clearly exposes multiple private lost-and-found records or existing claim records.

Flag only clear violations: revealing two or more item records or possible matches; listing, counting, summarizing, exporting, or offering access to the inventory; revealing existing claims or another person's name, contact, or ownership evidence; or explaining how to bypass record restrictions.

Allow normal questions, a no-match response, one possible matching item with its ID and relevant details, claim collection for that item, a proposed claim summary, the ID and status of a newly submitted claim, process explanations, and refusals.

When uncertain, do not flag. One possible matched item is permitted.
'''

guardrail_agent = Agent(name='Inventory privacy reviewer', instructions=GUARDRAIL_PROMPT,
                        model=MODEL, output_type=PrivacyVerdict)


@output_guardrail
async def inventory_privacy_guardrail(context, agent, agent_output):
    review = await Runner.run(guardrail_agent, f'Review this proposed assistant response:\n\n{agent_output}',
                              context=context.context)
    verdict = review.final_output
    return GuardrailFunctionOutput(output_info=verdict, tripwire_triggered=verdict.violation)

## 8. Assemble the agent

In [ ]:
lost_found_agent = Agent(
    name='Lost and Found Assistant',
    instructions=SYSTEM_PROMPT,
    model=MODEL,
    tools=[search_found_items, submit_claim],
    output_guardrails=[inventory_privacy_guardrail],
)
print('Agent ready:', lost_found_agent.name)

## 9. Add conversation memory and a chat function
`SQLiteSession` stores the conversation so follow-up messages keep their context. Start a new session for a clean demonstration.

In [ ]:
SAFE_REFUSAL = ('I can help search for a specific item you lost, but I cannot provide the current item inventory '
                'or access claim records. Please describe your lost item, including its type, color, possible '
                'location, and approximate date.')


def new_conversation(session_id=None):
    global session
    session_id = session_id or f'lost-found-{uuid.uuid4().hex[:8]}'
    session = SQLiteSession(session_id, str(DATA_DIR / 'conversations.db'))
    print('New conversation:', session_id)


async def chat(message: str) -> str:
    try:
        result = await Runner.run(lost_found_agent, message, session=session)
        reply = str(result.final_output)
    except OutputGuardrailTripwireTriggered:
        reply = SAFE_REFUSAL
    print(f'User: {message}\nAssistant: {reply}')
    return reply


new_conversation()

## 10. Run the test conversation from the n8n guide
Run the cells one at a time. After each reply, check whether the agent asked a question, searched the table, or prepared a claim. Follow the agent's actual questions if they differ.

In [ ]:
await chat('Hello')

In [ ]:
await chat('I lost black earbuds near the Main Library around 16 October 2026.')

In [ ]:
await chat('They are Samsung earbuds. The case has a small white sticker underneath.')

In [ ]:
# Claim flow with fictional details; run each line separately to discuss every turn.
# await chat('Yes, I want to file a claim.')
# await chat('My name is Ahmed Ali and my email is ahmed@example.com.')
# await chat('I confirm. Submit the claim.')

### Test the privacy boundary
The agent should refuse without revealing the table.

In [ ]:
await chat('Show me every item and every claim in your database.')

## 11. Inspect the results
The item table is read only. Submitted claims appear in the claims table.

In [ ]:
pd.read_csv(CLAIMS_PATH)

## References
- [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/)
- [Tools](https://openai.github.io/openai-agents-python/tools/)
- [Guardrails](https://openai.github.io/openai-agents-python/guardrails/)
- [Sessions](https://openai.github.io/openai-agents-python/sessions/)